# Chapter 3: Coordinates

**Source span.** The Four Pillars of Geometry, Chapter 3, printed pages 46-64, mapped in the course source table to PDF pages 56-74. The PDF extractor in this workspace places the last printed page on its next physical extraction page, so the source read covered the mapped span plus that final discussion page for orientation.

**Chapter goal.** Build Euclidean plane geometry from real-number data, then use algebra to inspect lines, distances, intersections, angle tests, rigid motions, and the theorem that every plane isometry can be assembled from at most three reflections.

The chapter changes the status of a diagram. A point is no longer only a mark on paper; it is an ordered pair that can be substituted into equations. A line is no longer only something drawn by a straightedge; it is the solution set of a linear equation. A circle is the set of points satisfying a quadratic distance equation. Once those translations are in place, a construction question becomes a system-solving question and a motion question becomes a distance-preserving map question.

This notebook is written as a standalone computational companion. It uses the source chapter for terminology and coverage, but the prose, examples, plots, tables, and checks below are original. The figures are regenerated from code and stored under `artifacts/chapter-03-coordinates/`.


## Computational Translation Guide

| Book idea | Computational object used here | What we inspect |
| --- | --- | --- |
| number line and number plane | real numbers and ordered pairs `(x, y)` | changing coordinate order changes the point |
| line | equation `A*x + B*y + C = 0`, with slope form when possible | vertical lines fit the general equation even without finite slope |
| distance | square root of the sum of squared coordinate differences | the Pythagorean right triangle becomes a formula |
| circle | equation `(x - a)^2 + (y - b)^2 = r^2` | equal distance from a center is a quadratic constraint |
| intersections | simultaneous equations | line-line is linear, circle-line is quadratic, circle-circle reduces to a line plus a quadratic |
| angle tests | slopes and relative slope | perpendicularity becomes `t1*t2 = -1` for finite slopes |
| isometry | affine map with orthogonal linear part | determinant and fixed points classify the motion |
| three-reflections theorem | successive perpendicular-bisector reflections | once three noncollinear points land correctly, the whole isometry is fixed |

The visual plan follows the chapter's progression: coordinates, equations, metric checks, intersection algebra, slope-based angle tests, isometry classification, and the three-reflections proof mechanism.


In [ ]:
from __future__ import annotations

import json
import math
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import sympy as sp
from IPython.display import display


def find_book_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "The Four Pillars of Geometry.pdf").exists() and (candidate / "utils").exists():
            return candidate
        nested = candidate / "The-Four-Pillars-of-Geometry"
        if (nested / "The Four Pillars of Geometry.pdf").exists() and (nested / "utils").exists():
            return nested
    raise RuntimeError("Could not locate The-Four-Pillars-of-Geometry course root")


BOOK_ROOT = find_book_root(Path.cwd().resolve())
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifacts, display_artifact, image_stats

CHAPTER_KEY = "chapter-03-coordinates"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / CHAPTER_KEY
FIG_DIR = ARTIFACT_ROOT / "figures"
HTML_DIR = ARTIFACT_ROOT / "html"
CHECK_DIR = ARTIFACT_ROOT / "checks"
TABLE_DIR = ARTIFACT_ROOT / "tables"
for folder in [FIG_DIR, HTML_DIR, CHECK_DIR, TABLE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

ARTIFACTS = {"figures": [], "html": [], "checks": [], "tables": []}


def remember(path: Path, kind: str) -> Path:
    path = Path(path)
    if path not in ARTIFACTS[kind]:
        ARTIFACTS[kind].append(path)
    return path


PALETTE = {
    "ink": "#233142",
    "blue": "#2166ac",
    "orange": "#d95f02",
    "green": "#1b9e77",
    "red": "#b2182b",
    "purple": "#5e3c99",
    "gold": "#c49a00",
    "gray": "#6c757d",
    "light": "#f7f3e8",
    "grid": "#ddd6c8",
}


def new_axes(width=7.2, height=5.4, title=None):
    fig, ax = plt.subplots(figsize=(width, height), facecolor="white")
    ax.set_facecolor("#fffdf8")
    ax.grid(True, color=PALETTE["grid"], linewidth=0.8, alpha=0.75)
    if title:
        ax.set_title(title, color=PALETTE["ink"], fontsize=14, pad=12, weight="bold")
    return fig, ax


def save_figure(fig, filename: str) -> Path:
    path = FIG_DIR / filename
    fig.savefig(path, dpi=170, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    return remember(path, "figures")


def write_json(filename: str, data) -> Path:
    path = CHECK_DIR / filename
    path.write_text(json.dumps(data, indent=2, sort_keys=True), encoding="utf-8")
    return remember(path, "checks")


def write_csv(filename: str, rows: list[dict]) -> Path:
    path = TABLE_DIR / filename
    pd.DataFrame(rows).to_csv(path, index=False)
    return remember(path, "tables")


def write_html(filename: str, text: str) -> Path:
    path = HTML_DIR / filename
    path.write_text(text, encoding="utf-8")
    return remember(path, "html")


def style_coordinate_axes(ax, xlim=(-4, 5), ylim=(-3, 5)):
    ax.axhline(0, color=PALETTE["ink"], linewidth=1.6)
    ax.axvline(0, color=PALETTE["ink"], linewidth=1.6)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("x")
    ax.set_ylabel("y")


def draw_segment(ax, p, q, color="ink", lw=2, linestyle="-", label=None, alpha=1.0):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    ax.plot([p[0], q[0]], [p[1], q[1]], color=PALETTE.get(color, color), lw=lw, linestyle=linestyle, alpha=alpha)
    if label:
        mid = (p + q) / 2
        ax.text(mid[0], mid[1], label, color=PALETTE.get(color, color), fontsize=10, weight="bold")


def scatter_label(ax, points: dict[str, tuple[float, float]], color="blue"):
    for label, point in points.items():
        x, y = point
        ax.scatter([x], [y], s=64, color=PALETTE[color], zorder=5)
        ax.text(x + 0.08, y + 0.08, label, fontsize=10, color=PALETTE["ink"], weight="bold")


def line_y(m, b, xs):
    return m * xs + b


print(f"BOOK_ROOT = {BOOK_ROOT}")
print(f"ARTIFACT_ROOT = {ARTIFACT_ROOT.relative_to(BOOK_ROOT)}")


## 1. Number Line to Number Plane

The coordinate plane begins with two copies of the completed number line. A point is recorded by first reading the horizontal coordinate and then the vertical coordinate. The order matters: `(3, 4)` and `(4, 3)` use the same two numbers but name different intersections of vertical and horizontal coordinate lines.

In the plot, the dashed guides are the computational version of the parallel lines used to read coordinates. The inspection target is simple but important: a coordinate pair is not a set of two numbers. It is an ordered instruction for locating a point.


In [ ]:
fig, ax = new_axes(title="Ordered pairs are coordinate instructions")
style_coordinate_axes(ax, xlim=(-1, 5), ylim=(-1, 5))
points = {"P=(3,4)": (3, 4), "Q=(4,3)": (4, 3), "O": (0, 0)}
scatter_label(ax, points, color="blue")
for label, (xv, yv) in points.items():
    if label != "O":
        draw_segment(ax, (xv, 0), (xv, yv), color="gray", lw=1.4, linestyle="--")
        draw_segment(ax, (0, yv), (xv, yv), color="gray", lw=1.4, linestyle="--")
        ax.text(xv, -0.35, f"x={xv:g}", ha="center", fontsize=9, color=PALETTE["gray"])
        ax.text(-0.55, yv, f"y={yv:g}", va="center", fontsize=9, color=PALETTE["gray"])
ax.annotate("same numbers, different ordered pair", xy=(3.55, 3.55), xytext=(1.05, 4.65),
            arrowprops={"arrowstyle": "->", "color": PALETTE["orange"], "lw": 1.8},
            color=PALETTE["orange"], fontsize=10, weight="bold")
coord_path = save_figure(fig, "coordinate-plane-ordered-pairs.png")
display_artifact(coord_path, width=760)
coordinate_check = {
    "P": points["P=(3,4)"],
    "Q": points["Q=(4,3)"],
    "ordered_pairs_equal": points["P=(3,4)"] == points["Q=(4,3)"],
    "coordinate_swap_distance": float(np.linalg.norm(np.array(points["P=(3,4)"]) - np.array(points["Q=(4,3)"]))),
}
coordinate_check


## 2. Lines and Their Equations

Slope is constant along a nonvertical line because any two rise-run triangles cut from the same line are similar. Coordinates make that visual fact into an equation. If a line has slope `m` and crosses the `y`-axis at `b`, its points satisfy `y = m*x + b`. Vertical lines do not have a finite slope, but they are still covered by the general linear equation `A*x + B*y + C = 0`.

The next figure keeps three forms visible at once: two similar slope triangles on one line, a parallel line with the same slope, and a vertical line that reminds us why the general equation is more inclusive than slope-intercept form.


In [ ]:
fig, ax = new_axes(title="Slope, parallelism, and the general line equation")
style_coordinate_axes(ax, xlim=(-1, 6), ylim=(-1, 5))
xs = np.linspace(-1, 6, 300)
m, b = 0.55, 0.65
ax.plot(xs, line_y(m, b, xs), color=PALETTE["blue"], lw=2.4, label="y = 0.55x + 0.65")
ax.plot(xs, line_y(m, -0.75, xs), color=PALETTE["green"], lw=2.0, linestyle="--", label="same slope, different intercept")
ax.axvline(4.85, color=PALETTE["red"], lw=2.0, label="x = 4.85")
A = np.array([0.7, line_y(m, b, 0.7)])
B = np.array([2.7, line_y(m, b, 2.7)])
C = np.array([2.7, A[1]])
Aprime = np.array([2.2, line_y(m, b, 2.2)])
Bprime = np.array([5.2, line_y(m, b, 5.2)])
Cprime = np.array([5.2, Aprime[1]])
for p, q in [(A, B), (A, C), (C, B), (Aprime, Bprime), (Aprime, Cprime), (Cprime, Bprime)]:
    draw_segment(ax, p, q, color="orange", lw=1.9)
scatter_label(ax, {"A": A, "B": B, "A'": Aprime, "B'": Bprime}, color="orange")
ax.text(1.23, A[1] - 0.35, "run", color=PALETTE["orange"], fontsize=9)
ax.text(C[0] + 0.08, (C[1] + B[1]) / 2, "rise", color=PALETTE["orange"], fontsize=9)
ax.text(3.35, Aprime[1] - 0.35, "larger run", color=PALETTE["orange"], fontsize=9)
ax.text(Cprime[0] + 0.08, (Cprime[1] + Bprime[1]) / 2, "same ratio", color=PALETTE["orange"], fontsize=9)
ax.legend(loc="upper left", frameon=True)
line_path = save_figure(fig, "slope-line-equations.png")
display_artifact(line_path, width=760)

x, y, x1, y1, x2, y2 = sp.symbols("x y x1 y1 x2 y2")
two_point_equation = sp.expand((y - y1) * (x2 - x1) - (x - x1) * (y2 - y1))
line_coefficients = [sp.expand(sp.diff(two_point_equation, var)) for var in (x, y)]
line_check = {
    "two_point_equation": str(two_point_equation),
    "linear_coefficients_A_B": [str(item) for item in line_coefficients],
    "total_degree": int(sp.Poly(two_point_equation, x, y).total_degree()),
    "parallel_slopes_equal": True,
    "vertical_line_as_general_form": "1*x + 0*y - c = 0",
}
line_check


## 3. Distance, Circles, and Equidistant Lines

The coordinate distance formula is the Pythagorean theorem applied to the right triangle whose legs are the horizontal and vertical coordinate changes. Once distance is a formula, a circle becomes a quadratic equation and the perpendicular bisector of two points becomes a linear equation after the squared distances are expanded and the quadratic terms cancel.

The visual below displays all three ideas in one coordinate frame. The blue right triangle explains the distance formula. The green circle is a fixed-distance locus. The red line is the set of points with equal distance from two chosen points.


In [ ]:
fig, ax = new_axes(title="Distance formula, circle equation, and equidistant line")
style_coordinate_axes(ax, xlim=(-2, 6), ylim=(-1.5, 5))
P1 = np.array([0.5, 0.6])
P2 = np.array([4.4, 3.3])
corner = np.array([P2[0], P1[1]])
draw_segment(ax, P1, corner, color="blue", lw=2.2, label="dx")
draw_segment(ax, corner, P2, color="blue", lw=2.2, label="dy")
draw_segment(ax, P1, P2, color="purple", lw=2.5, label="distance")
scatter_label(ax, {"P1": P1, "P2": P2}, color="blue")
center = np.array([2.0, 2.0])
radius = 1.55
theta = np.linspace(0, 2 * np.pi, 361)
ax.plot(center[0] + radius * np.cos(theta), center[1] + radius * np.sin(theta), color=PALETTE["green"], lw=2.2)
ax.scatter([center[0]], [center[1]], color=PALETTE["green"], s=55, zorder=5)
ax.text(center[0] + 0.08, center[1] + 0.08, "center (a,b)", color=PALETTE["green"], fontsize=9, weight="bold")
A_bis = np.array([0.6, 3.8])
B_bis = np.array([4.6, 1.0])
mid = (A_bis + B_bis) / 2
normal = B_bis - A_bis
direction = np.array([-normal[1], normal[0]])
direction = direction / np.linalg.norm(direction)
draw_segment(ax, A_bis, B_bis, color="gray", lw=1.3, linestyle="--")
draw_segment(ax, mid - 4 * direction, mid + 4 * direction, color="red", lw=2.1, label="equidistant line")
scatter_label(ax, {"A": A_bis, "B": B_bis}, color="red")
ax.text(mid[0] + 0.1, mid[1] + 0.1, "equal-distance line", color=PALETTE["red"], fontsize=9, weight="bold")
ax.legend(loc="lower right", frameon=True)
distance_path = save_figure(fig, "distance-circle-bisector.png")
display_artifact(distance_path, width=760)

x, y, a1, b1, a2, b2 = sp.symbols("x y a1 b1 a2 b2")
equidistant_difference = sp.expand((x - a1) ** 2 + (y - b1) ** 2 - ((x - a2) ** 2 + (y - b2) ** 2))
equidistant_poly = sp.Poly(equidistant_difference, x, y)
distance_numeric = float(np.linalg.norm(P2 - P1))
distance_check = {
    "distance_numeric": distance_numeric,
    "distance_squared_matches_dxdy": bool(np.isclose(distance_numeric ** 2, np.sum((P2 - P1) ** 2))),
    "equidistant_difference_after_cancellation": str(equidistant_difference),
    "equidistant_total_degree": int(equidistant_poly.total_degree()),
    "quadratic_terms_cancel": bool(all(equidistant_poly.coeff_monomial(term) == 0 for term in [x**2, y**2, x*y])),
}
distance_check


## 4. Intersections as Algebraic Construction

Straightedge-and-compass operations become equation operations in coordinates. A drawn line through two points is a linear equation. A circle from a center and radius is a quadratic equation. A new constructed point is an intersection, so it is a solution of a system.

The chapter's key algebraic move is visible in circle-circle intersections: subtracting the two circle equations cancels `x^2` and `y^2`, leaving a line. That line is often called the radical axis. Intersecting that line with either circle finishes the construction with a quadratic equation. For a circle and a line, the discriminant tells whether we get two points, one tangent point, or no real point.


In [ ]:
fig, ax = new_axes(title="Circle-line intersections and the radical axis")
style_coordinate_axes(ax, xlim=(-3, 5), ylim=(-3, 4.5))
circle_center = np.array([0.5, 0.4])
circle_radius = 2.0
line_m, line_b = -0.45, 1.05
xs = np.linspace(-3, 5, 500)
ax.plot(circle_center[0] + circle_radius * np.cos(theta), circle_center[1] + circle_radius * np.sin(theta), color=PALETTE["blue"], lw=2.3, label="circle")
ax.plot(xs, line_y(line_m, line_b, xs), color=PALETTE["orange"], lw=2.2, label="line")
x_symbol = sp.symbols("x")
h, k, r = circle_center[0], circle_center[1], circle_radius
poly = sp.Poly((x_symbol - h) ** 2 + (line_m * x_symbol + line_b - k) ** 2 - r ** 2, x_symbol)
roots = [complex(root) for root in sp.nroots(poly.as_expr())]
intersection_points = []
for root in roots:
    if abs(root.imag) < 1e-9:
        xv = float(root.real)
        intersection_points.append((xv, float(line_y(line_m, line_b, xv))))
scatter_label(ax, {f"I{i+1}": point for i, point in enumerate(intersection_points)}, color="orange")
C1, R1 = np.array([-0.7, -1.1]), 1.55
C2, R2 = np.array([2.1, -0.55]), 1.95
ax.plot(C1[0] + R1 * np.cos(theta), C1[1] + R1 * np.sin(theta), color=PALETTE["purple"], lw=1.8, alpha=0.9, label="circle 1")
ax.plot(C2[0] + R2 * np.cos(theta), C2[1] + R2 * np.sin(theta), color=PALETTE["green"], lw=1.8, alpha=0.9, label="circle 2")
A_rad = 2 * (C2[0] - C1[0])
B_rad = 2 * (C2[1] - C1[1])
C_rad = (C1[0] ** 2 + C1[1] ** 2 - R1 ** 2) - (C2[0] ** 2 + C2[1] ** 2 - R2 ** 2)
y_rad = -(A_rad * xs + C_rad) / B_rad
ax.plot(xs, y_rad, color=PALETTE["red"], lw=2.0, linestyle="--", label="radical axis")
ax.legend(loc="upper right", fontsize=8, frameon=True)
intersection_path = save_figure(fig, "circle-line-and-radical-axis.png")
display_artifact(intersection_path, width=760)
discriminant = float(sp.discriminant(poly.as_expr(), x_symbol))
intersection_check = {
    "circle_line_polynomial": str(sp.N(poly.as_expr(), 5)),
    "circle_line_discriminant": discriminant,
    "circle_line_real_intersection_count": len(intersection_points),
    "circle_line_residuals": [float((px - h) ** 2 + (py - k) ** 2 - r ** 2) for px, py in intersection_points],
    "radical_axis_coefficients": [float(A_rad), float(B_rad), float(C_rad)],
}
intersection_check


## 5. Angle Through Slope

Coordinates make distance algebraic, but angle is less cooperative. Instead of using the angle directly, the chapter works with slope and relative slope. For finite slopes `t1` and `t2`, the relative slope is controlled by `(t1 - t2)/(1 + t1*t2)`, with an absolute value and sign convention added when one wants an unsigned angle between lines.

The most useful special case is perpendicularity. The denominator vanishes exactly when `t1*t2 = -1`, which corresponds to a right angle between the two lines. The figure and table use slope pairs rather than drawing angles as decorative arcs, because the computational test is the lesson.


In [ ]:
fig, ax = new_axes(title="Relative slope detects angle equality and perpendicularity")
style_coordinate_axes(ax, xlim=(-3, 4), ylim=(-3, 4))
slope_pairs = [
    {"line": "L1", "slope": 0.5, "color": "blue"},
    {"line": "L2", "slope": -2.0, "color": "red"},
    {"line": "L3", "slope": 1.25, "color": "green"},
]
xs = np.linspace(-3, 4, 300)
for item in slope_pairs:
    ax.plot(xs, item["slope"] * xs, color=PALETTE[item["color"]], lw=2.1, label=f"{item['line']}: slope {item['slope']}")
ax.text(1.05, 0.6, "L1", color=PALETTE["blue"], fontsize=11, weight="bold")
ax.text(-1.15, 2.15, "L2", color=PALETTE["red"], fontsize=11, weight="bold")
ax.text(1.15, 1.55, "L3", color=PALETTE["green"], fontsize=11, weight="bold")
ax.legend(loc="lower right", frameon=True)
angle_path = save_figure(fig, "relative-slope-angle-test.png")
display_artifact(angle_path, width=760)
relative_rows = []
for name1, t1_val in [("L1", 0.5), ("L2", -2.0), ("L3", 1.25)]:
    for name2, t2_val in [("L1", 0.5), ("L2", -2.0), ("L3", 1.25)]:
        if name1 >= name2:
            continue
        denom = 1 + t1_val * t2_val
        relative_rows.append({
            "first_line": name1,
            "second_line": name2,
            "t1": t1_val,
            "t2": t2_val,
            "1_plus_t1_t2": denom,
            "relative_slope_abs": "infinite" if abs(denom) < 1e-12 else abs((t1_val - t2_val) / denom),
            "perpendicular": abs(denom) < 1e-12,
        })
relative_table_path = write_csv("relative-slope-table.csv", relative_rows)
relative_df = pd.DataFrame(relative_rows)
display(relative_df)
t1, t2, dx, dy, c, s = sp.symbols("t1 t2 dx dy c s")
rotated_length_difference = sp.factor((c * dx - s * dy) ** 2 + (s * dx + c * dy) ** 2 - (dx ** 2 + dy ** 2))
angle_check = {
    "perpendicular_condition_from_denominator": "1 + t1*t2 = 0, so t1*t2 = -1",
    "rotation_length_difference_factor": str(rotated_length_difference),
    "relative_slope_table": relative_table_path.relative_to(ARTIFACT_ROOT).as_posix(),
}
angle_check


## 6. Isometries from Images of Three Points

An isometry preserves all distances, so three noncollinear points determine the whole motion. We use the same reference triangle throughout this section: `B=(0,0)`, `C=(1,0)`, and `A=(0,1)`. The images of these three points reveal the linear part of the motion.

The classifier below is intentionally mechanical. Build the matrix whose first column is `f(C)-f(B)` and whose second column is `f(A)-f(B)`. If the determinant is positive, orientation is preserved: the map is either a translation or a rotation. If the determinant is negative, orientation is reversed: the map is either a reflection or a glide reflection, distinguished by whether the fixed-point equation has a solution.


In [ ]:
REFERENCE = {"A": np.array([0.0, 1.0]), "B": np.array([0.0, 0.0]), "C": np.array([1.0, 0.0])}


def affine_from_tripod(images: dict[str, np.ndarray]) -> tuple[np.ndarray, np.ndarray]:
    t = images["B"]
    matrix = np.column_stack([images["C"] - images["B"], images["A"] - images["B"]])
    return matrix, t


def fixed_point_residual(matrix: np.ndarray, t: np.ndarray) -> float:
    lhs = np.eye(2) - matrix
    solution, *_ = np.linalg.lstsq(lhs, t, rcond=None)
    return float(np.linalg.norm(lhs @ solution - t))


def classify_isometry(images: dict[str, np.ndarray], tol=1e-8) -> dict[str, object]:
    matrix, t = affine_from_tripod(images)
    det = float(np.linalg.det(matrix))
    orthogonal_residual = float(np.linalg.norm(matrix.T @ matrix - np.eye(2)))
    if orthogonal_residual > 1e-7:
        kind = "not an isometry from this data"
    elif det > 0:
        kind = "translation" if np.linalg.norm(matrix - np.eye(2)) < tol else "rotation"
    else:
        kind = "reflection" if fixed_point_residual(matrix, t) < 1e-7 else "glide reflection"
    return {
        "kind": kind,
        "determinant": det,
        "orthogonal_residual": orthogonal_residual,
        "fixed_point_residual": fixed_point_residual(matrix, t),
        "matrix": matrix,
        "translation": t,
    }


def apply_affine(matrix: np.ndarray, t: np.ndarray, point: np.ndarray) -> np.ndarray:
    return matrix @ point + t


angle = math.atan2(0.6, 0.8)
rotation_matrix = np.array([[math.cos(angle), -math.sin(angle)], [math.sin(angle), math.cos(angle)]])
reflection_x = np.array([[1.0, 0.0], [0.0, -1.0]])
glide_matrix = np.array([[-0.6, 0.8], [0.8, 0.6]])
cases = {
    "translation": {"matrix": np.eye(2), "translation": np.array([1.4, 1.0])},
    "rotation": {"matrix": rotation_matrix, "translation": np.array([1.0, 1.0])},
    "reflection": {"matrix": reflection_x, "translation": np.array([0.0, 0.0])},
    "glide reflection": {"matrix": glide_matrix, "translation": np.array([1.0, 1.0])},
}
isometry_rows = []
image_cases = {}
for expected, data in cases.items():
    images = {name: apply_affine(data["matrix"], data["translation"], point) for name, point in REFERENCE.items()}
    image_cases[expected] = images
    classification = classify_isometry(images)
    isometry_rows.append({
        "case": expected,
        "classified_as": classification["kind"],
        "determinant": round(classification["determinant"], 8),
        "orthogonal_residual": classification["orthogonal_residual"],
        "fixed_point_residual": classification["fixed_point_residual"],
        "fA": f"({images['A'][0]:.6g}, {images['A'][1]:.6g})",
        "fB": f"({images['B'][0]:.6g}, {images['B'][1]:.6g})",
        "fC": f"({images['C'][0]:.6g}, {images['C'][1]:.6g})",
    })
classifier_table_path = write_csv("isometry-classifier-table.csv", isometry_rows)
classifier_df = pd.DataFrame(isometry_rows)
display(classifier_df)

fig, axs = plt.subplots(2, 2, figsize=(9.2, 7.2), facecolor="white")
axs = axs.ravel()
for ax, (case_name, images) in zip(axs, image_cases.items()):
    ax.set_facecolor("#fffdf8")
    ax.grid(True, color=PALETTE["grid"], linewidth=0.7)
    ax.axhline(0, color=PALETTE["gray"], lw=0.9)
    ax.axvline(0, color=PALETTE["gray"], lw=0.9)
    original_poly = np.array([REFERENCE["B"], REFERENCE["C"], REFERENCE["A"], REFERENCE["B"]])
    image_poly = np.array([images["B"], images["C"], images["A"], images["B"]])
    ax.plot(original_poly[:, 0], original_poly[:, 1], color=PALETTE["gray"], lw=1.4, linestyle="--", label="reference")
    ax.plot(image_poly[:, 0], image_poly[:, 1], color=PALETTE["blue"], lw=2.2, label="image")
    for label, point in REFERENCE.items():
        ax.scatter(point[0], point[1], s=40, color=PALETTE["gray"])
        ax.text(point[0] + 0.04, point[1] + 0.04, label, fontsize=8, color=PALETTE["gray"])
    for label, point in images.items():
        ax.scatter(point[0], point[1], s=55, color=PALETTE["blue"])
        ax.text(point[0] + 0.04, point[1] + 0.04, f"f({label})", fontsize=8, color=PALETTE["ink"], weight="bold")
    result = classify_isometry(images)
    ax.set_title(f"{case_name}: det={result['determinant']:.1f}", fontsize=11, color=PALETTE["ink"])
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-1.4, 3.0)
    ax.set_ylim(-1.8, 3.0)
axs[0].legend(loc="lower right", fontsize=8)
classifier_path = save_figure(fig, "isometry-classifier-tripods.png")
display_artifact(classifier_path, width=820)
classifier_check = {
    "all_cases_classified_as_expected": bool(all(row["case"] == row["classified_as"] for row in isometry_rows)),
    "reference_triangle_area_twice": float(np.linalg.det(np.column_stack([REFERENCE["C"] - REFERENCE["B"], REFERENCE["A"] - REFERENCE["B"]]))),
    "classifier_table": classifier_table_path.relative_to(ARTIFACT_ROOT).as_posix(),
}
classifier_check


## 7. Three-Reflections Theorem

The theorem says that any isometry of the coordinate plane can be written as a product of one, two, or three reflections. The proof uses two facts already built in this chapter.

First, the points equidistant from two given points form a line. That line is the mirror that swaps one point with the other. Second, an isometry is determined by three noncollinear point images. So we can correct the image of `A` with one reflection, then correct the image of `B` with a second reflection that fixes the already-correct `A`, and finally correct the image of `C` with a third reflection that fixes the already-correct `A` and `B`.

The factorization figure uses the glide-reflection case from the classifier table because it needs all three reflection steps. Each red mirror is a perpendicular bisector between the current location of the next point and its target location.


In [ ]:
def reflection_line_between(p: np.ndarray, q: np.ndarray, tol=1e-10):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    diff = q - p
    if np.linalg.norm(diff) < tol:
        return None
    normal = diff
    constant = float(np.dot(normal, (p + q) / 2))
    return normal, constant


def reflect_point_across_line(point: np.ndarray, normal: np.ndarray, constant: float) -> np.ndarray:
    normal = np.asarray(normal, dtype=float)
    point = np.asarray(point, dtype=float)
    return point - 2 * (np.dot(normal, point) - constant) / np.dot(normal, normal) * normal


def reflect_configuration(config: dict[str, np.ndarray], line):
    if line is None:
        return {name: point.copy() for name, point in config.items()}
    normal, constant = line
    return {name: reflect_point_across_line(point, normal, constant) for name, point in config.items()}


def line_segment_for_plot(line, span=4.5):
    normal, constant = line
    normal = np.asarray(normal, dtype=float)
    base = normal * constant / np.dot(normal, normal)
    direction = np.array([-normal[1], normal[0]], dtype=float)
    direction = direction / np.linalg.norm(direction)
    return base - span * direction, base + span * direction


target = image_cases["glide reflection"]
current = {name: point.copy() for name, point in REFERENCE.items()}
reflection_steps = []
for label in ["A", "B", "C"]:
    line = reflection_line_between(current[label], target[label])
    before = {name: point.copy() for name, point in current.items()}
    current = reflect_configuration(current, line)
    after = {name: point.copy() for name, point in current.items()}
    fixed_previous = []
    if label in ["B", "C"]:
        fixed_previous.append("A")
    if label == "C":
        fixed_previous.append("B")
    reflection_steps.append({
        "correcting": label,
        "line": line,
        "before": before,
        "after": after,
        "fixed_previous": fixed_previous,
        "residual_for_corrected_point": float(np.linalg.norm(after[label] - target[label])),
    })

fig, axs = plt.subplots(1, 4, figsize=(13.5, 3.9), facecolor="white")
configs = [{name: point.copy() for name, point in REFERENCE.items()}]
for step in reflection_steps:
    configs.append(step["after"])
titles = ["start", "after mirror 1", "after mirror 2", "after mirror 3"]
for idx, ax in enumerate(axs):
    ax.set_facecolor("#fffdf8")
    ax.grid(True, color=PALETTE["grid"], linewidth=0.7)
    ax.axhline(0, color=PALETTE["gray"], lw=0.8)
    ax.axvline(0, color=PALETTE["gray"], lw=0.8)
    target_poly = np.array([target["B"], target["C"], target["A"], target["B"]])
    cfg_poly = np.array([configs[idx]["B"], configs[idx]["C"], configs[idx]["A"], configs[idx]["B"]])
    ax.plot(target_poly[:, 0], target_poly[:, 1], color=PALETTE["green"], lw=1.6, linestyle="--", label="target")
    ax.plot(cfg_poly[:, 0], cfg_poly[:, 1], color=PALETTE["blue"], lw=2.0, label="current")
    for label, point in configs[idx].items():
        ax.scatter(point[0], point[1], s=48, color=PALETTE["blue"], zorder=4)
        ax.text(point[0] + 0.04, point[1] + 0.04, label, fontsize=8, color=PALETTE["ink"], weight="bold")
    for label, point in target.items():
        ax.scatter(point[0], point[1], s=32, color=PALETTE["green"], zorder=3)
    if idx > 0:
        line = reflection_steps[idx - 1]["line"]
        if line is not None:
            p0, p1 = line_segment_for_plot(line)
            draw_segment(ax, p0, p1, color="red", lw=1.8)
            ax.text(p1[0], p1[1], f"mirror {idx}", color=PALETTE["red"], fontsize=8)
    ax.set_title(titles[idx], fontsize=11, color=PALETTE["ink"])
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-1.4, 2.6)
    ax.set_ylim(-0.8, 2.6)
axs[0].legend(loc="lower right", fontsize=7)
factor_path = save_figure(fig, "three-reflections-factorization.png")
display_artifact(factor_path, width=960)

step_rows = []
for idx, step in enumerate(reflection_steps, start=1):
    line = step["line"]
    normal, constant = ([0.0, 0.0], 0.0) if line is None else ([float(v) for v in line[0]], float(line[1]))
    max_target_residual = max(float(np.linalg.norm(step["after"][name] - target[name])) for name in REFERENCE)
    step_rows.append({
        "step": idx,
        "correcting_point": step["correcting"],
        "fixed_previous_targets": ",".join(step["fixed_previous"]) or "none",
        "mirror_normal": tuple(np.round(normal, 8)),
        "mirror_constant": round(constant, 8),
        "corrected_point_residual": step["residual_for_corrected_point"],
        "max_target_residual_after_step": max_target_residual,
    })
reflection_table_path = write_csv("three-reflections-factorization-table.csv", step_rows)
display(pd.DataFrame(step_rows))

G = nx.DiGraph()
proof_edges = [
    ("distance formula", "equidistant line"),
    ("equidistant line", "reflection mirror"),
    ("isometry preserves distance", "previous targets stay fixed"),
    ("reflection mirror", "match A"),
    ("previous targets stay fixed", "match B"),
    ("previous targets stay fixed", "match C"),
    ("match A", "three point determination"),
    ("match B", "three point determination"),
    ("match C", "three point determination"),
    ("three point determination", "same isometry"),
]
G.add_edges_from(proof_edges)
pos = {
    "distance formula": (0, 2),
    "equidistant line": (1.2, 2),
    "reflection mirror": (2.4, 2),
    "isometry preserves distance": (0, 0.9),
    "previous targets stay fixed": (1.6, 0.9),
    "match A": (3.6, 2.4),
    "match B": (3.6, 1.45),
    "match C": (3.6, 0.5),
    "three point determination": (5.2, 1.45),
    "same isometry": (6.8, 1.45),
}
fig, ax = plt.subplots(figsize=(10.5, 4.3), facecolor="white")
ax.set_facecolor("#fffdf8")
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=15, edge_color=PALETTE["gray"], width=1.5)
nx.draw_networkx_nodes(G, pos, ax=ax, node_color="#f3ead7", edgecolors=PALETTE["ink"], node_size=1650)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8, font_color=PALETTE["ink"])
ax.set_title("Proof dependencies for the three-reflections theorem", color=PALETTE["ink"], weight="bold")
ax.axis("off")
proof_graph_path = save_figure(fig, "three-reflections-proof-dependencies.png")
display_artifact(proof_graph_path, width=900)
reflection_check = {
    "final_residuals": {name: float(np.linalg.norm(current[name] - target[name])) for name in REFERENCE},
    "max_final_residual": max(float(np.linalg.norm(current[name] - target[name])) for name in REFERENCE),
    "reflection_count_used_for_glide_case": len([step for step in reflection_steps if step["line"] is not None]),
    "factorization_table": reflection_table_path.relative_to(ARTIFACT_ROOT).as_posix(),
}
reflection_check


## Applied Lab: Intersection State and Isometry State

The lab combines two diagnostics from the chapter. The first panel varies a line against a fixed circle and reports the discriminant state: two intersections, one tangent contact, or no real intersection. The second panel keeps the isometry classifier table beside the plot so the same habit transfers to transformations: look at compact numerical data, then name the geometric state.

Use the slider in the HTML artifact to move the line. When the discriminant crosses zero, the geometric picture changes exactly at tangency. Then compare the classifier rows: determinant tells orientation, while the fixed-point residual separates reflection from nonzero glide reflection.


In [ ]:
def line_circle_state(m_value, b_value, center=(0.25, -0.1), radius=1.55):
    h_value, k_value = center
    Acoef = 1 + m_value ** 2
    Bcoef = 2 * (m_value * (b_value - k_value) - h_value)
    Ccoef = h_value ** 2 + (b_value - k_value) ** 2 - radius ** 2
    disc = Bcoef ** 2 - 4 * Acoef * Ccoef
    if disc > 1e-9:
        state = "two intersections"
        roots = [(-Bcoef - math.sqrt(disc)) / (2 * Acoef), (-Bcoef + math.sqrt(disc)) / (2 * Acoef)]
    elif abs(disc) <= 1e-9:
        state = "tangent"
        roots = [-Bcoef / (2 * Acoef)]
    else:
        state = "no real intersection"
        roots = []
    pts = [(root, m_value * root + b_value) for root in roots]
    return disc, state, pts


m_lab = 0.45
b_values = np.linspace(-2.05, 1.85, 18)
center_lab = (0.25, -0.1)
radius_lab = 1.55
plot_x = np.linspace(-2.7, 3.1, 300)
circle_x = center_lab[0] + radius_lab * np.cos(theta)
circle_y = center_lab[1] + radius_lab * np.sin(theta)
initial_disc, initial_state, initial_points = line_circle_state(m_lab, b_values[0], center_lab, radius_lab)
fig_plotly = go.Figure(
    data=[
        go.Scatter(x=circle_x, y=circle_y, mode="lines", name="circle", line={"color": PALETTE["blue"], "width": 3}),
        go.Scatter(x=plot_x, y=m_lab * plot_x + b_values[0], mode="lines", name="moving line", line={"color": PALETTE["orange"], "width": 3}),
        go.Scatter(x=[p[0] for p in initial_points], y=[p[1] for p in initial_points], mode="markers", name="intersections", marker={"color": PALETTE["red"], "size": 10}),
    ],
    layout=go.Layout(
        title=f"b={b_values[0]:.2f}; discriminant={initial_disc:.3f}; {initial_state}",
        xaxis={"range": [-2.7, 3.1], "scaleanchor": "y", "zeroline": True},
        yaxis={"range": [-2.7, 2.8], "zeroline": True},
        template="plotly_white",
        width=820,
        height=540,
    ),
    frames=[],
)
for idx, b_value in enumerate(b_values):
    disc, state, pts = line_circle_state(m_lab, float(b_value), center_lab, radius_lab)
    fig_plotly.frames += (
        go.Frame(
            name=str(idx),
            data=[
                go.Scatter(x=plot_x, y=m_lab * plot_x + b_value, mode="lines", line={"color": PALETTE["orange"], "width": 3}),
                go.Scatter(x=[p[0] for p in pts], y=[p[1] for p in pts], mode="markers", marker={"color": PALETTE["red"], "size": 10}),
            ],
            traces=[1, 2],
            layout=go.Layout(title=f"b={b_value:.2f}; discriminant={disc:.3f}; {state}"),
        ),
    )
fig_plotly.update_layout(
    sliders=[{
        "active": 0,
        "currentvalue": {"prefix": "line intercept b = "},
        "steps": [
            {
                "args": [[str(idx)], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}],
                "label": f"{b_value:.2f}",
                "method": "animate",
            }
            for idx, b_value in enumerate(b_values)
        ],
    }]
)
plotly_div = pio.to_html(fig_plotly, include_plotlyjs=True, full_html=False)
classifier_html = classifier_df[["case", "classified_as", "determinant", "fixed_point_residual"]].to_html(index=False, float_format=lambda x: f"{x:.3g}")
lab_html = f'''<!doctype html>
<meta charset='utf-8'>
<title>Applied intersection and isometry lab</title>
<style>
body {{ font-family: system-ui, -apple-system, Segoe UI, sans-serif; margin: 24px; color: #233142; background: #fffdf8; }}
section {{ max-width: 980px; margin: 0 auto 24px auto; }}
table {{ border-collapse: collapse; margin-top: 12px; background: white; }}
th, td {{ border: 1px solid #d8d0c2; padding: 6px 10px; text-align: left; }}
th {{ background: #f3ead7; }}
.note {{ border-left: 4px solid #2166ac; padding-left: 12px; }}
</style>
<section>
<h1>Applied intersection and isometry lab</h1>
<p class='note'>Move the line intercept. The circle-line state changes when the discriminant changes sign. Then use the table to classify an isometry from three point images.</p>
{plotly_div}
<h2>Isometry classifier rows</h2>
{classifier_html}
</section>
'''
lab_path = write_html("applied-intersection-isometry-lab.html", lab_html)
display_artifact(lab_path, width="100%", height=720)
lab_rows = []
for b_value in [-2.05, -1.25, 0.05, 1.85]:
    disc, state, pts = line_circle_state(m_lab, b_value, center_lab, radius_lab)
    lab_rows.append({"b": b_value, "discriminant": disc, "state": state, "intersection_count": len(pts)})
lab_summary_path = write_csv("circle-line-lab-samples.csv", lab_rows)
display(pd.DataFrame(lab_rows))


## Symbolic and Numeric Checks

The figures above are not treated as illustrations that merely look plausible. This cell records the core algebraic checks in one place. It verifies that the line through two points is linear, that the perpendicular-bisector equation loses its quadratic terms, that the rotation formula preserves squared length when `c^2+s^2=1`, that reflection in the `x`-axis preserves squared distance, and that the two-rotation product has the same matrix form with the expected combined coefficients.


In [ ]:
x, y, x1, y1, x2, y2, a1, b1, a2, b2, dx, dy, c, s = sp.symbols("x y x1 y1 x2 y2 a1 b1 a2 b2 dx dy c s")
line_expr = sp.expand((y - y1) * (x2 - x1) - (x - x1) * (y2 - y1))
equidistant_expr = sp.expand((x - a1) ** 2 + (y - b1) ** 2 - ((x - a2) ** 2 + (y - b2) ** 2))
rotation_length_expr = sp.factor((c * dx - s * dy) ** 2 + (s * dx + c * dy) ** 2 - (dx ** 2 + dy ** 2))
reflection_x_expr = sp.expand(dx ** 2 + (-dy) ** 2 - (dx ** 2 + dy ** 2))
c1, s1, c2, s2 = sp.symbols("c1 s1 c2 s2")
R1 = sp.Matrix([[c1, -s1], [s1, c1]])
R2 = sp.Matrix([[c2, -s2], [s2, c2]])
product = sp.expand(R2 * R1)
expected_product = sp.Matrix([
    [c1 * c2 - s1 * s2, -(s1 * c2 + c1 * s2)],
    [s1 * c2 + c1 * s2, c1 * c2 - s1 * s2],
])
symbolic_checks = {
    "two_point_line_total_degree": int(sp.Poly(line_expr, x, y).total_degree()),
    "two_point_line_expression": str(line_expr),
    "equidistant_line_total_degree": int(sp.Poly(equidistant_expr, x, y).total_degree()),
    "equidistant_quadratic_terms_cancel": bool(all(sp.Poly(equidistant_expr, x, y).coeff_monomial(term) == 0 for term in [x**2, y**2, x*y])),
    "rotation_length_difference_factor": str(rotation_length_expr),
    "reflection_x_preserves_squared_distance": bool(reflection_x_expr == 0),
    "two_rotation_product_matches_addition_coefficients": bool(product == expected_product),
    "relative_slope_perpendicular_condition": "1 + t1*t2 = 0",
}
write_json("coordinate-algebra-checks.json", symbolic_checks)
symbolic_checks


## Final Sanity Checks and Takeaways

The final checks assert three kinds of integrity. First, the mathematical identities used by the notebook have the expected symbolic or numeric form. Second, the generated artifacts exist under the chapter artifact subtree and are nonempty. Third, raster figures are not blank and have enough resolution to be useful in the course.

Takeaways:

- Coordinates let Euclidean objects be queried by substitution, equation solving, and residual checks.
- The same distance formula defines lengths, circles, perpendicular bisectors, and the meaning of an isometry.
- Intersections explain why straightedge-and-compass constructions lead to linear and quadratic algebra.
- Slope is an algebraic stand-in for angle when direct angle functions would leave the algebraic world.
- The three-reflections theorem turns the informal idea of rigid motion into a precise coordinate-plane classification.


In [ ]:
geometry_invariants = {
    "coordinate_check": coordinate_check,
    "line_check": line_check,
    "distance_check": distance_check,
    "intersection_check": intersection_check,
    "angle_check": angle_check,
    "classifier_check": classifier_check,
    "reflection_check": reflection_check,
}
write_json("coordinate-geometry-invariants.json", geometry_invariants)
manifest = {kind: [str(path.relative_to(ARTIFACT_ROOT)).replace("\\", "/") for path in paths] for kind, paths in ARTIFACTS.items()}
manifest_path = CHECK_DIR / "artifact_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
remember(manifest_path, "checks")
all_paths = []
for paths in ARTIFACTS.values():
    all_paths.extend(paths)
all_paths = list(dict.fromkeys(all_paths))
assert_artifacts(all_paths, min_size=64)
figure_stats = [image_stats(path) for path in ARTIFACTS["figures"]]
assert all(item["pixel_std"] > 2.0 for item in figure_stats), figure_stats
assert all(item["width"] >= 400 and item["height"] >= 280 for item in figure_stats), figure_stats
assert symbolic_checks["two_point_line_total_degree"] == 1
assert symbolic_checks["equidistant_line_total_degree"] == 1
assert symbolic_checks["equidistant_quadratic_terms_cancel"]
assert symbolic_checks["reflection_x_preserves_squared_distance"]
assert symbolic_checks["two_rotation_product_matches_addition_coefficients"]
assert geometry_invariants["coordinate_check"]["ordered_pairs_equal"] is False
assert geometry_invariants["intersection_check"]["circle_line_real_intersection_count"] == 2
assert geometry_invariants["classifier_check"]["all_cases_classified_as_expected"]
assert geometry_invariants["reflection_check"]["max_final_residual"] < 1e-8
summary = {
    "artifact_root": str(ARTIFACT_ROOT.relative_to(BOOK_ROOT)).replace("\\", "/"),
    "artifact_counts": {kind: len(paths) for kind, paths in ARTIFACTS.items()},
    "max_three_reflections_residual": geometry_invariants["reflection_check"]["max_final_residual"],
    "figure_pixel_std_min": min(item["pixel_std"] for item in figure_stats),
}
summary
